# Lab 9 · Reading drills: two levels deep

**Today:** two reading drills on nested draws, then PS3.

**Before you start:** labs 7 and 8. Short on purpose.

Each section names one idea, explains what it does, and asks you to **predict
what a cell prints before you run it**. Write the prediction down, on paper, out
loud, or in a comment. A prediction you can compare against the output is what
tells you which parts of the code you can already read.

Most sections end with a **Test your understanding** task: write a small piece
of code, then run the check cell under it. Every task has a hint in the **Hints**
block at the end of the notebook, for when you want it. The check never grades and never
breaks anything. A ⬜ means not attempted yet, a ❌ means not yet and comes with
a hint, and a ✅ means passing. Run the check cells rather than editing them.
Everything else in the notebook is yours to change.

**AI in this lab.** Until your prediction is written down, work at level 1, with
no AI. The prediction is how you find out what you can read unaided, and both
exams are level 1. Once you have run a cell, level 3 is encouraged: ask your
tutor to explain anything you missed.

Run every cell, and change things to see what happens. Nothing in this notebook
can be broken in a way that matters.

## 1 · Which line is the hidden layer?

**Before running: this loop has three random draws in it. Which one is the hidden layer, the setting that is never observed directly, and how many times does each line run?**

In [ ]:
import numpy as np

rng = np.random.default_rng(73)
totals = []
for _ in range(4):
    efficiency = rng.beta(8, 2)          # line A
    total = 0
    for _ in range(3):
        made = rng.poisson(50)           # line B
        kept = rng.binomial(made, efficiency)   # line C
        total += kept
    totals.append(total)
print(totals)

Line A runs 4 times, once per outer pass, and is the hidden layer: each batch draws its own efficiency, and all three inner runs share it. Lines B and C run 12 times each. Reading a nested simulation means asking, for each random line, which loop owns it, and therefore what stays fixed while what varies.

**Test your understanding.** Section 1's loop shares one efficiency across a batch's three runs. Write a function named `independent_total` that takes one parameter, `rng`, a generator, and simulates one batch in which the efficiency is drawn **inside** the inner loop, so each run gets a fresh efficiency and none is shared. Keep `rng.beta(8, 2)` for the efficiency, `rng.poisson(50)` for `made`, and `rng.binomial(made, efficiency)` for `kept`. It should return one whole number, the batch's total of `kept` over its three runs. The checks use seeded generators, so the draw order matters. This is question 1. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named independent_total(rng), efficiency drawn per run, 3 runs of poisson(50)
import numpy as np

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("independent_total", expect=140, args=(np.random.default_rng(79),),
      hint="per run: efficiency = rng.beta(8, 2), made = rng.poisson(50), kept = rng.binomial(made, efficiency) — in that order, three times")
check("independent_total", expect=115, args=(np.random.default_rng(83),))

## 2 · Spot the planted mismatch

A colleague claims this chunk simulates "2,000 days, each day's rate drawn fresh, then one count per day." **Read it against the claim before running. One line makes the claim false. Which?**

In [ ]:
import numpy as np

rng = np.random.default_rng(89)
rate = rng.gamma(5, 4)
counts = []
for _ in range(2000):
    counts.append(rng.poisson(rate))
print("mean", round(np.mean(counts), 1), " sd", round(np.std(counts), 1))

The `rate` line sits **outside** the loop: drawn once and shared by all 2,000 days, so there is one hidden setting rather than 2,000 fresh ones. The printed sd shows it (near √rate, which is one-level noise). The indentation states the claim.

**Test your understanding.** Write a function named `fresh_rate_sd` that takes two parameters: `n_days`, the number of days to simulate, and `rng`, a generator. It should fix section 2's chunk to match the colleague's claim: draw the rate with `rng.gamma(5, 4)` inside the loop, once per day, then draw one poisson count from that rate, and return the sd of the counts rounded to 1 decimal. Predict first: will the sd come out bigger or smaller than the printed one? This is question 2. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named fresh_rate_sd(n_days, rng)
import numpy as np

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("fresh_rate_sd", expect=10.1, args=(3000, np.random.default_rng(97)),
      hint="rate inside the loop; gamma(5, 4) then poisson(rate), each day")

## If you finish early

PS3 is due Friday.

## If you are stuck

Wave someone over. This hour exists so that a stuck step costs you a minute
rather than an evening. Known snags:

- **`independent_total` misses the seed checks.** The draw order inside each run must be exactly beta, poisson, binomial. Any other order pulls different numbers off the stream.
- **Section 2's fix prints nearly the same sd as the broken version on your seed.** Compare against the reasoning instead: a shared rate adds no day-to-day drift, and a fresh rate must widen the spread.

## Hints

**Question 1 · `independent_total`.** Copy the inside of section 1's outer loop into the function body, without the outer loop and without `totals`. Set `total = 0`, then write a `for _ in range(3):` loop whose body draws `efficiency`, then `made`, then `kept`, in that order, and adds `kept` to `total`. Return `total`. Moving the `efficiency` line into the inner loop is the change that makes each run's efficiency fresh.

**Question 2 · `fresh_rate_sd`.** Section 2's chunk becomes the function body once the `rate` line moves inside the loop. Make an empty list, then write a `for _ in range(n_days):` loop whose body draws `rate = rng.gamma(5, 4)` and appends `rng.poisson(rate)`. After the loop, return `round(np.std(counts), 1)`. The indentation of the `rate` line decides whether there is one hidden setting or `n_days` of them.